## What is it?

LLM-based chunking uses a language model to intelligently determine where to split text based on semantic understanding. The LLM analyzes the document and decides optimal chunk boundaries, potentially generating summaries or extracting key information.

This is like having an intelligent editor who understands the content and knows exactly where topics begin and end.

In [1]:
text = """Artificial intelligence is transforming technology and shaping the future.
Machine learning algorithms are becoming more sophisticated every day.
Deep learning models can now process vast amounts of data efficiently.
Neural networks are inspired by the human brain's structure.
The best pasta recipes include fresh ingredients and proper cooking techniques.
Italian cuisine emphasizes quality olive oil and regional cheeses.
Authentic carbonara uses guanciale, eggs, pecorino romano, and black pepper.
Cooking pasta al dente ensures the best texture and flavor.
Climate change is affecting ecosystems worldwide.
Rising temperatures are causing glaciers to melt at unprecedented rates.
Scientists warn that immediate action is needed to reduce carbon emissions.
Renewable energy sources offer hope for a sustainable future."""

## Advantages

- Most intelligent and context-aware splitting
- Can add contextual summaries to chunks
- Understands semantic boundaries better than rules
- Can adapt to different document types
- Improves retrieval quality significantly
- Can extract key information

## Disadvantages

- Very expensive (LLM API calls for every chunk)
- Slowest processing time
- Requires API access and costs money
- Not deterministic (results may vary)
- Overkill for simple documents
- Latency issues for large documents
- Complex to implement and maintain

In [3]:
from langchain_ollama import ChatOllama
from pydantic import BaseModel
from langchain_core.prompts import ChatPromptTemplate


In [4]:
# pydantic class for structured output

class Chunk(BaseModel):
    chunk_text: str
    summary: str


class Chunker(BaseModel):
    chunks: list[Chunk]

In [5]:
# define model

model = ChatOllama(
    model="smollm2:360m",
    temperature=0
)

llm_chunker = model.with_structured_output(
    schema=Chunker
)

In [6]:
# prompt for chunking

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """You are an expert Text Chunker that splits the given text
into meaningful chunks.

Rules:
- Split the text according to natural topic boundaries.
- Do not change the original text.
- Do not remove any information.
- Keep related sentences together.
- For every chunk, generate a 1-2 line summary.
"""
    ),
    (
        "human",
        "Split the given text into chunks.\n\nText: {text}"
    )
])

In [7]:
# chunking through LLM

model_chain = prompt | llm_chunker

response = model_chain.invoke({
    "text": text
})

In [8]:
response

Chunker(chunks=[Chunk(chunk_text='Artificial intelligence is transforming technology and shaping the future.', summary='AI is changing the way technology works and is expected to continue to do so in the future.'), Chunk(chunk_text='Machine learning algorithms are becoming more sophisticated every day.', summary='Machine learning algorithms are getting better and better at processing data.'), Chunk(chunk_text='Deep learning models can now process vast amounts of data efficiently.', summary='Deep learning models are getting better at processing large amounts of data and are becoming more efficient.'), Chunk(chunk_text="Neural networks are inspired by the human brain's structure.", summary='Neural networks are getting better at understanding how the brain works and are getting better at processing data.'), Chunk(chunk_text='The best pasta recipes include fresh ingredients and proper cooking techniques.', summary='The best pasta recipes include fresh ingredients and proper cooking techniq

In [10]:
# create document of chunks
from langchain_core.documents import Document

docs = [
    Document(
        page_content=chunk.chunk_text,
        metadata={
            "summary": chunk.summary
        }
    )
    for chunk in response.chunks
]

In [11]:
docs

[Document(metadata={'summary': 'AI is changing the way technology works and is expected to continue to do so in the future.'}, page_content='Artificial intelligence is transforming technology and shaping the future.'),
 Document(metadata={'summary': 'Machine learning algorithms are getting better and better at processing data.'}, page_content='Machine learning algorithms are becoming more sophisticated every day.'),
 Document(metadata={'summary': 'Deep learning models are getting better at processing large amounts of data and are becoming more efficient.'}, page_content='Deep learning models can now process vast amounts of data efficiently.'),
 Document(metadata={'summary': 'Neural networks are getting better at understanding how the brain works and are getting better at processing data.'}, page_content="Neural networks are inspired by the human brain's structure."),
 Document(metadata={'summary': 'The best pasta recipes include fresh ingredients and proper cooking techniques.'}, page_